# Hybrid Retrieval: Combining Dense and Sparse Search

Dense retrieval compares embeddings and can match related meanings even when the query and a passage use different words. Sparse retrieval, such as BM25, matches terms and is often strong for exact names, codes, and shared vocabulary. **Hybrid retrieval** combines both signals to find results that either method alone might miss.

This notebook builds a tiny dense retriever and BM25 retriever, then combines their result lists using Reciprocal Rank Fusion (RRF). RRF uses rank positions rather than adding raw scores, which matters because BM25 and cosine similarity have different scales and meanings. Everything runs with the Python standard library; a hand-designed encoder is used for transparent illustration, not as a production model.

## Learning goals

By the end, you should be able to:

- Explain why dense and sparse retrievers can complement each other.
- Generate independent dense and BM25 rankings for the same query and corpus.
- Explain why directly adding raw scores from different retrievers is unsafe without calibration.
- Implement Reciprocal Rank Fusion and understand its rank constant and weights.
- Compare dense-only, sparse-only, and hybrid results with relevance judgments.
- Recognize the cost, duplicate, candidate-coverage, and evaluation trade-offs of hybrid retrieval.

## 1. What is hybrid retrieval?

A retriever returns ranked candidates for a query. Dense and sparse retrieval use different evidence:

| Method | Main signal | Often useful for | Common limitation |
| --- | --- | --- | --- |
| Dense | Similarity between learned text vectors | Paraphrases and semantic similarity | Can blur exact identifiers or rank topical non-answers |
| Sparse / BM25 | Weighted overlap of query and document terms | Names, identifiers, and shared terminology | Can miss synonyms and paraphrases with little term overlap |

Hybrid retrieval runs both methods and combines their candidate lists or scores. This can improve the chance that useful evidence is found through at least one signal. It does not automatically improve quality: the methods can share failure modes, return irrelevant chunks, or duplicate results. Measure the combined ranking on representative labeled queries.

### The retrieval pipeline

For a query, the workflow in this notebook is:

1. Embed the query and rank corpus chunks by cosine similarity.
2. Tokenize the query and rank chunks by BM25.
3. Merge the two ranked lists with a fusion rule.
4. Select the final top-K chunks for downstream use.

Each retriever can return its own top-N candidates. The union of those lists is the fusion candidate set. If a relevant chunk is absent from both lists, rank fusion cannot restore it; fusion improves ordering only among candidates it receives.

## 2. Shared corpus and queries

We reuse the same stable chunk IDs from Notebooks 01–03. The examples include a semantic paraphrase, an exact-term query, and an identifier query. A tiny hand-designed concept encoder below makes the dense mechanics inspectable and has the same limitations as the earlier dense notebook: it is not learned and only connects concepts explicitly listed in its mapping.

In [1]:
corpus = [
    {'chunk_id': 'C01', 'source': 'account_help', 'topic': 'password',
     'text': 'To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.'},
    {'chunk_id': 'C02', 'source': 'account_help', 'topic': 'security',
     'text': 'For account security, turn on two-factor authentication and never share your verification code with anyone.'},
    {'chunk_id': 'C03', 'source': 'shopping_policy', 'topic': 'refunds',
     'text': 'Refunds for eligible orders are returned to the original payment method after the returned item is inspected.'},
    {'chunk_id': 'C04', 'source': 'shipping_guide', 'topic': 'delivery',
     'text': 'Standard shipping usually takes five to seven business days. Express shipping is available at checkout.'},
    {'chunk_id': 'C05', 'source': 'shopping_policy', 'topic': 'returns',
     'text': 'Unopened products can be returned within thirty days of delivery. Include the order number in your return request.'},
    {'chunk_id': 'C06', 'source': 'tenant_handbook', 'topic': 'lease',
     'text': 'The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.'},
    {'chunk_id': 'C07', 'source': 'cycling_guide', 'topic': 'safety',
     'text': 'Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.'},
    {'chunk_id': 'C08', 'source': 'account_help', 'topic': 'support',
     'text': 'Contact support if you cannot access the email address linked to your account.'},
    {'chunk_id': 'C09', 'source': 'food_guide', 'topic': 'allergies',
     'text': 'Read the ingredient label carefully if you have a food allergy. Contact the manufacturer when allergen information is unclear.'},
    {'chunk_id': 'C10', 'source': 'account_help', 'topic': 'limitations',
     'text': 'Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.'},
]

queries = {
    'paraphrase': 'How can I access my account after forgetting the login secret?',
    'exact_terms': 'password reset link',
    'identifier': 'support contact email account',
}

for label, query in queries.items():
    print(f'{label}: {query}')
print(f'\nCorpus chunks: {len(corpus)}')

paraphrase: How can I access my account after forgetting the login secret?
exact_terms: password reset link
identifier: support contact email account

Corpus chunks: 10


## 3. Build the dense retriever

This tiny teaching encoder maps a few explicit word groups to concept dimensions. Related words share dimensions, such as `password`, `login`, and `secret`. It demonstrates the mechanics of embedding and cosine ranking without installing a model or downloading files. It is essentially a manually authored synonym map—not a learned embedding model and not suitable for production.

In [2]:
import math
import re
from collections import Counter, defaultdict

DIMENSIONS = [
    'credentials', 'recovery', 'email_support', 'delivery', 'refund',
    'returns', 'lease', 'bicycle', 'allergy',
]
DIMENSION_INDEX = {name: index for index, name in enumerate(DIMENSIONS)}

CONCEPTS = {
    'password': 'credentials', 'login': 'credentials', 'secret': 'credentials',
    'reset': 'recovery', 'forgot': 'recovery', 'forgetting': 'recovery',
    'recover': 'recovery', 'recovery': 'recovery', 'regain': 'recovery',
    'email': 'email_support', 'support': 'email_support', 'contact': 'email_support',
    'shipping': 'delivery', 'delivery': 'delivery', 'refund': 'refund',
    'refunds': 'refund', 'return': 'returns', 'returned': 'returns',
    'lease': 'lease', 'deposit': 'lease', 'helmet': 'bicycle',
    'bicycle': 'bicycle', 'allergy': 'allergy', 'allergen': 'allergy',
}

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def normalize(vector):
    norm = math.sqrt(sum(value * value for value in vector))
    if norm == 0:
        return [0.0] * len(vector)
    return [value / norm for value in vector]

def teaching_encode(text):
    vector = [0.0] * len(DIMENSIONS)
    terms = set(tokenize(text))
    for term in terms:
        concept = CONCEPTS.get(term)
        if concept is not None:
            vector[DIMENSION_INDEX[concept]] += 1.0
    return normalize(vector)

def cosine_similarity(left, right):
    if len(left) != len(right):
        raise ValueError('Vectors must have equal dimensions')
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return sum(a * b for a, b in zip(left, right)) / (left_norm * right_norm)

dense_vectors = {chunk['chunk_id']: teaching_encode(chunk['text']) for chunk in corpus}
print('Dimensions:', len(DIMENSIONS))
print('Query vector:', [round(x, 3) for x in teaching_encode(queries['paraphrase'])])

Dimensions: 9
Query vector: [0.894, 0.447, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


Each chunk vector is precomputed once. For every query, the retriever encodes the query, compares it with every chunk vector, then sorts by cosine similarity descending. A tie is broken by chunk ID so results are repeatable. This is an exact scan over ten chunks, not an approximate-nearest-neighbor index.

In [3]:
def dense_rank(query, chunks=corpus, vectors=dense_vectors):
    query_vector = teaching_encode(query)
    results = []
    for chunk in chunks:
        results.append({
            **chunk,
            'score': cosine_similarity(query_vector, vectors[chunk['chunk_id']]),
        })
    return sorted(results, key=lambda item: (-item['score'], item['chunk_id']))

for item in dense_rank(queries['paraphrase'])[:5]:
    print(f"{item['chunk_id']} cosine={item['score']:.3f} | {item['text']}")

C10 cosine=0.800 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.
C01 cosine=0.730 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
C02 cosine=0.000 | For account security, turn on two-factor authentication and never share your verification code with anyone.
C03 cosine=0.000 | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
C04 cosine=0.000 | Standard shipping usually takes five to seven business days. Express shipping is available at checkout.


## 4. Build the sparse retriever with BM25

We now build a minimal inverted index and BM25 implementation. The index maps each term to the chunks that contain it. BM25 weights rarer terms more strongly, saturates repeated term frequency, and normalizes for chunk length. The formula below uses the same positive IDF variant as Notebook 03; real search libraries may implement different details or query analyzers.

In [4]:
doc_tf = {}
doc_lengths = {}
postings = defaultdict(list)

for chunk in corpus:
    terms = tokenize(chunk['text'])
    tf = Counter(terms)
    doc_tf[chunk['chunk_id']] = tf
    doc_lengths[chunk['chunk_id']] = len(terms)
    for term in sorted(tf):
        postings[term].append(chunk['chunk_id'])

N = len(corpus)
average_length = sum(doc_lengths.values()) / N

def bm25_idf(term):
    df = len(postings.get(term, []))
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

def bm25_score(query, chunk_id, k1=1.5, b=0.75):
    if k1 < 0 or not 0 <= b <= 1:
        raise ValueError('Require k1 >= 0 and 0 <= b <= 1')
    score = 0.0
    length = doc_lengths[chunk_id]
    for term in set(tokenize(query)):
        frequency = doc_tf[chunk_id].get(term, 0)
        if frequency == 0:
            continue
        denominator = frequency + k1 * (1 - b + b * length / average_length)
        score += bm25_idf(term) * frequency * (k1 + 1) / denominator
    return score

def sparse_rank(query, chunks=corpus, k1=1.5, b=0.75):
    results = []
    for chunk in chunks:
        score = bm25_score(query, chunk['chunk_id'], k1=k1, b=b)
        if score > 0:
            results.append({**chunk, 'score': score})
    return sorted(results, key=lambda item: (-item['score'], item['chunk_id']))

for item in sparse_rank(queries['paraphrase'])[:5]:
    print(f"{item['chunk_id']} BM25={item['score']:.3f} | {item['text']}")

C08 BM25=4.023 | Contact support if you cannot access the email address linked to your account.
C10 BM25=2.988 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.
C05 BM25=2.375 | Unopened products can be returned within thirty days of delivery. Include the order number in your return request.
C03 BM25=2.077 | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
C07 BM25=1.776 | Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.


## 5. Inspect where the retrievers agree and disagree

Before combining results, compare the independent rankings. Their disagreement is the reason fusion may help. A result appearing in both lists may be supported by two kinds of evidence; a result appearing in only one list can still be valuable if that retriever is right for the query.

In [5]:
def show_rankings(query, depth=5):
    dense = dense_rank(query)[:depth]
    sparse = sparse_rank(query)[:depth]
    print('Query:', query)
    print('Dense:', [(x['chunk_id'], round(x['score'], 3)) for x in dense])
    print('BM25: ', [(x['chunk_id'], round(x['score'], 3)) for x in sparse])
    dense_ids = {x['chunk_id'] for x in dense}
    sparse_ids = {x['chunk_id'] for x in sparse}
    print('Shared:', sorted(dense_ids & sparse_ids))
    print('Dense only:', sorted(dense_ids - sparse_ids))
    print('BM25 only:', sorted(sparse_ids - dense_ids))

for label, query in queries.items():
    print(f'\n--- {label} ---')
    show_rankings(query)


--- paraphrase ---
Query: How can I access my account after forgetting the login secret?
Dense: [('C10', 0.8), ('C01', 0.73), ('C02', 0.0), ('C03', 0.0), ('C04', 0.0)]
BM25:  [('C08', 4.023), ('C10', 2.988), ('C05', 2.375), ('C03', 2.077), ('C07', 1.776)]
Shared: ['C03', 'C10']
Dense only: ['C01', 'C02', 'C04']
BM25 only: ['C05', 'C07', 'C08']

--- exact_terms ---
Query: password reset link
Dense: [('C10', 0.949), ('C01', 0.866), ('C02', 0.0), ('C03', 0.0), ('C04', 0.0)]
BM25:  [('C01', 5.762), ('C10', 2.822)]
Shared: ['C01', 'C10']
Dense only: ['C02', 'C03', 'C04']
BM25 only: []

--- identifier ---
Query: support contact email account
Dense: [('C08', 1.0), ('C09', 0.447), ('C01', 0.408), ('C02', 0.0), ('C03', 0.0)]
BM25:  [('C08', 6.972), ('C09', 1.445), ('C01', 1.347), ('C02', 1.205), ('C10', 1.091)]
Shared: ['C01', 'C02', 'C08', 'C09']
Dense only: ['C03']
BM25 only: ['C10']


A dense ranking includes all chunks in this exact-scan example, even chunks with a zero vector similarity. BM25 returns only chunks with at least one lexical match. A real dense vector database may also return a limited candidate set, often with a similarity threshold. Candidate generation settings need to be understood before comparing or fusing rankings.

## 6. Why not add the raw scores?

A tempting approach is `dense_cosine + BM25`. But raw scores have different scales and distributions:

- Cosine similarity is bounded by -1 and 1 for nonzero vectors, although observed ranges vary by model and corpus.
- BM25 scores are nonnegative in this implementation and are not bounded by 1; they depend on corpus statistics, query terms, chunk lengths, and parameters.
- A change in the embedding model or BM25 configuration can shift one score distribution without changing the other.

Raw addition therefore makes the result depend heavily on arbitrary score scale and weighting. Alternatives include score normalization (with careful calibration), learning-to-rank on labeled data, or **rank fusion**, which combines ordering rather than raw magnitude. We will use Reciprocal Rank Fusion because it is simple and robust for this lesson.

### Reciprocal Rank Fusion (RRF)

For a chunk `d`, RRF adds a contribution from each ranked list where it appears:

`RRF(d) = Σ weight_i / (c + rank_i(d))`

where `rank_i(d)` is the 1-based rank in retriever i, `c` is a positive rank constant, and `weight_i` optionally weights each retriever. A chunk ranked highly by multiple systems accumulates support. A chunk returned by only one system can still rank well if its position is high.

The constant `c` reduces the difference between early ranks and later ranks. It controls how much the very top positions dominate; it is not learned relevance confidence. RRF scores are fusion scores, not probabilities.

In [ ]:
def reciprocal_rank_fusion(ranked_lists, weights=None, c=60, candidate_metadata=None):
    if c <= 0:
        raise ValueError('c must be positive')
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    if len(weights) != len(ranked_lists):
        raise ValueError('Provide exactly one weight per ranking')
    if any(weight < 0 for weight in weights):
        raise ValueError('weights must be non-negative')

    fusion_scores = defaultdict(float)
    for ranked_list, weight in zip(ranked_lists, weights):
        for rank, item in enumerate(ranked_list, start=1):
            fusion_scores[item['chunk_id']] += weight / (c + rank)

    metadata = candidate_metadata or {}
    fused = []
    for chunk_id, score in fusion_scores.items():
        record = dict(metadata.get(chunk_id, {'chunk_id': chunk_id}))
        record['rrf_score'] = score
        fused.append(record)
    return sorted(fused, key=lambda item: (-item['rrf_score'], item['chunk_id']))

query = queries['paraphrase']
dense_list = dense_rank(query)
sparse_list = sparse_rank(query)
metadata_by_id = {chunk['chunk_id']: chunk for chunk in corpus}
hybrid_results = reciprocal_rank_fusion(
    [dense_list, sparse_list],
    c=60,
    candidate_metadata=metadata_by_id,
)

for rank, item in enumerate(hybrid_results, start=1):
    print(f"{rank}. {item['chunk_id']} RRF={item['rrf_score']:.5f} | {item['text']}")

RRF's input lists are ranked, and each list contributes according to its position. Missing chunks contribute nothing from that list. Notice that a dense full scan assigns ranks even to zero-similarity chunks; a production dense retriever often returns only top-N results, so its candidate list and ranks will differ. Fusion behavior depends on the candidate depth and ranking policy for every retriever.

### Trace one fusion score by hand

If a chunk is rank 2 in dense results and rank 1 in sparse results, with equal weights and `c=60`, its RRF score is `1/(60+2) + 1/(60+1)`. A chunk ranked 1 by only one retriever gets `1/(60+1)`. The first chunk gets support from both lists, so its total is larger in this example.

In [ ]:
manual_score = 1 / (60 + 2) + 1 / (60 + 1)
single_list_score = 1 / (60 + 1)
print('Rank 2 + rank 1:', round(manual_score, 6))
print('Rank 1 in one list only:', round(single_list_score, 6))
assert manual_score > single_list_score

## 7. Candidate depth, top-K, and deduplication

A production retriever usually returns top-N candidates from each method, then fusion returns a final top-K. These are different settings:

- **Per-retriever candidate depth** controls how many results each method contributes to fusion.
- **Final K** controls how many fused results go to the next stage.

Fusion is keyed by chunk ID, so the same chunk returned by both retrievers is one result with contributions from both. If retrievers use different IDs for the same underlying content, deduplication needs a stable canonical ID or a separate similarity-based deduplication step. Increasing candidate depth can improve recall but also adds noise and computation.

In [ ]:
def top_n(ranked_list, n):
    if n < 0:
        raise ValueError('n must be non-negative')
    return ranked_list[:n]

def hybrid_rank(query, candidate_depth=5, c=60, weights=(1.0, 1.0)):
    dense_candidates = top_n(
        [item for item in dense_rank(query) if item['score'] > 0],
        candidate_depth,
    )
    sparse_candidates = top_n(sparse_rank(query), candidate_depth)
    fused = reciprocal_rank_fusion(
        [dense_candidates, sparse_candidates],
        weights=list(weights),
        c=c,
        candidate_metadata=metadata_by_id,
    )
    return fused

for depth in (1, 3, 5, 10):
    results = hybrid_rank(queries['exact_terms'], candidate_depth=depth)
    print(f'depth={depth}:', [(x['chunk_id'], round(x['rrf_score'], 5)) for x in results[:5]])

The dense demonstration scans every chunk, so `dense_rank` includes zero-similarity chunks for inspection. Before fusion, this example drops zero-score dense results because the hand-designed encoder has no evidence for those chunks. A real vector search often returns its nearest top-N even if scores are low, so choose a threshold or candidate policy deliberately. Select candidate depth based on latency, recall, and reranking capacity.

## 8. Compare dense-only, sparse-only, and hybrid retrieval

Use relevance judgments to compare rankings. This small dataset supports illustrative judgments for three queries. We will calculate precision@K and recall@K. The dense teaching encoder is deliberately simplistic, so the table teaches how to evaluate systems—not which production method is universally best.

In [ ]:
relevance = {
    'paraphrase': {'C01'},
    'exact_terms': {'C01'},
    'identifier': {'C08'},
}

def precision_at_k(results, relevant_ids, k, id_key='chunk_id'):
    if k <= 0:
        raise ValueError('k must be positive')
    selected = results[:k]
    if not selected:
        return 0.0
    hits = sum(item[id_key] in relevant_ids for item in selected)
    return hits / len(selected)

def recall_at_k(results, relevant_ids, k, id_key='chunk_id'):
    if k <= 0:
        raise ValueError('k must be positive')
    if not relevant_ids:
        raise ValueError('relevant_ids must not be empty')
    hits = sum(item[id_key] in relevant_ids for item in results[:k])
    return hits / len(relevant_ids)

def rankings_for(label):
    query_text = queries[label]
    return {
        'dense': dense_rank(query_text),
        'BM25': sparse_rank(query_text),
        'hybrid': hybrid_rank(query_text, candidate_depth=5),
    }

for label in queries:
    print(f'\n{label}: {queries[label]}')
    for method, results in rankings_for(label).items():
        print(f'  {method:6s} top 3: {[item["chunk_id"] for item in results[:3]]}')
        print(f'         P@3={precision_at_k(results, relevance[label], 3):.2f}, '
              f'R@3={recall_at_k(results, relevance[label], 3):.2f}')

Interpret metrics alongside the actual passages. For the paraphrase, BM25 retrieves chunks with accidental shared words while dense search finds both the useful passage and a misleading archived note; RRF can still put that misleading note first because it appears near the top of both lists. This is an intentional example of consensus reinforcing a shared error. A query with one relevant chunk can reach recall@3 as soon as that chunk appears in the first three results; precision@3 falls if the other two are irrelevant. A tiny hand-labeled set is useful for a lesson, but far too small to choose a production retriever or tune a fusion method.

### Rank fusion versus calibrated score fusion

RRF is not the only fusion approach. One could normalize scores and combine them, learn a ranking model from labels, or use a domain-specific weighted rule. These alternatives require care:

- **Min-max normalization** depends on the candidate set; adding one extreme score can change every normalized value.
- **Z-score normalization** can be unstable with small or skewed result sets.
- **Learned fusion** needs enough representative relevance labels and monitoring for drift.
- **RRF** ignores raw score margins: ranks 1 and 2 always differ by a fixed rank-based amount regardless of whether their raw scores were nearly tied or far apart.

RRF is a practical baseline because it avoids direct raw-score comparability, but evaluate it against alternatives on the actual task.

## 9. Optional retriever weighting

RRF permits weights so one retriever contributes more than another. Equal weights are a sensible starting point, not a universal optimum. If exact identifiers are especially important, sparse retrieval might deserve more weight; if user queries are usually paraphrases, dense retrieval might deserve more. Choose weights from labeled evaluation data and validate across query categories rather than tuning on one example.

In [ ]:
query = queries['paraphrase']
dense_candidates = top_n([item for item in dense_rank(query) if item['score'] > 0], 5)
sparse_candidates = top_n(sparse_rank(query), 5)

for weights in ((1.0, 1.0), (2.0, 1.0), (1.0, 2.0)):
    results = reciprocal_rank_fusion(
        [dense_candidates, sparse_candidates],
        weights=list(weights),
        c=60,
        candidate_metadata=metadata_by_id,
    )
    print(f'weights dense/BM25={weights}:',
          [(item['chunk_id'], round(item['rrf_score'], 5)) for item in results[:5]])

Weight changes can alter rankings, but a desired one-query outcome is not enough evidence for a configuration. A robust evaluation should include multiple query types and a held-out set, so tuning does not merely overfit the examples used during development.

## 10. Production considerations and failure modes

- **Latency and cost:** two retrievers require two search operations and potentially two embedding/index infrastructures. Parallel execution can reduce wall-clock latency, but adds operational complexity.
- **Candidate depth:** shallow per-method lists can omit relevant chunks before fusion; overly deep lists can dilute results and spend resources on noise.
- **Identity and deduplication:** fusion assumes IDs refer to the same content across systems. Keep canonical chunk IDs and metadata consistent.
- **Stale or inconsistent indexes:** update dense and sparse indexes together when source content changes; otherwise fusion may return conflicting versions.
- **Score and metric direction:** dense indexes may return distances where lower is better or similarities where higher is better. Convert each output to a rank consistently before fusion.
- **Common failure modes:** both methods may miss evidence due to bad chunking, missing source material, restrictive filters, or an unrepresentative embedding model.
- **Evaluation:** compare dense-only, sparse-only, and hybrid with the same judged queries; monitor latency and downstream answer grounding as well as ranking metrics.

Hybrid retrieval is an architecture choice, not a guaranteed quality upgrade. It is useful when dense and lexical signals provide complementary evidence and when the benefit justifies additional complexity.

## 11. Exercises

1. Pick a query where dense and BM25 rankings differ. Inspect why each method ranks its top result where it does.
2. Calculate manually the RRF score for a chunk ranked 1st in one list and 4th in another using `c=60` and equal weights.
3. Change per-retriever candidate depth from 1 to 10. Which relevant chunks become available to fusion, and which extra results are noise?
4. Use a query containing an exact identifier (add the identifier to one chunk first). Does BM25 or the dense teaching encoder rank it more reliably? Explain.
5. Change dense/BM25 weights and compare the top three results for every labeled query. Do not select weights based on only one query.
6. Why is adding raw BM25 and cosine scores not a sound default? Describe a situation where calibrated score fusion could be appropriate.
7. Design five additional queries, label relevant chunk IDs, and compare dense-only, BM25-only, and hybrid Precision@K/Recall@K.

### Optional extension: basic RRF invariants

These checks verify that a chunk returned by both systems receives contributions from both, and that invalid rank-fusion arguments are rejected.

In [ ]:
toy_a = [{'chunk_id': 'X'}, {'chunk_id': 'Y'}]
toy_b = [{'chunk_id': 'X'}, {'chunk_id': 'Z'}]
toy_fused = reciprocal_rank_fusion([toy_a, toy_b], c=10)
toy_scores = {item['chunk_id']: item['rrf_score'] for item in toy_fused}
assert toy_scores['X'] == 2 / 11
assert toy_scores['Y'] == 1 / 12
assert toy_scores['Z'] == 1 / 12
assert toy_fused[0]['chunk_id'] == 'X'

try:
    reciprocal_rank_fusion([toy_a], weights=[1.0], c=0)
except ValueError:
    print('Non-positive c is rejected as expected.')
else:
    raise AssertionError('c=0 should have raised ValueError')

print('RRF shared-result and ordering checks passed.')

## Summary

Hybrid retrieval combines complementary retrieval signals. Dense retrieval can bridge wording differences; BM25 can strongly match shared terms and exact identifiers. Reciprocal Rank Fusion merges ranked lists without assuming their raw scores share a scale. Its result depends on each retriever's candidate depth, ranking, weight, and rank constant.

Fusion cannot recover a chunk no retriever returned, and adding a second system increases cost and operational work. Compare dense-only, sparse-only, and hybrid rankings on the same representative relevance judgments, inspect actual passages, and measure downstream answer quality. The next lesson moves from combining search signals to narrowing search with **metadata filters**.